In [5]:
# Installing Kaggle Libraries
!pip install -q kaggle


Uploading Kaggle JSON file

In [6]:
!mkdir -p ~/.kaggle
!echo "KGAT_551cfaa0e9db74588a6abc3a3bb101e0" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
!pip install -q kaggle
!kaggle competitions list

ref                                                                              deadline             category         reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge      2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                               2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                       2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection             2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3          

In [55]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [56]:
!mkdir -p "/content/drive/MyDrive/Twitter-Sentiment-Analysis/data"
!mkdir -p "/content/drive/MyDrive/Twitter-Sentiment-Analysis/models"
!mkdir -p "/content/drive/MyDrive/Twitter-Sentiment-Analysis/notebooks"

In [57]:
!mv training.1600000.processed.noemoticon.csv \
"/content/drive/MyDrive/Twitter-Sentiment-Analysis/data/"

In [58]:
import pickle

pickle.dump(model,
open("/content/drive/MyDrive/Twitter-Sentiment-Analysis/models/sentiment_model.pkl","wb"))

In [7]:
!kaggle datasets download -d kazanova/sentiment140
!unzip -o sentiment140.zip
!ls

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
sentiment140.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  sentiment140.zip
  inflating: training.1600000.processed.noemoticon.csv  
sample_data	  sentiment_data
sentiment140.zip  training.1600000.processed.noemoticon.csv


In [8]:
!rm -rf /content/sentiment_data
!mkdir /content/sentiment_data

!unzip -o sentiment140.zip -d /content/sentiment_data

Archive:  sentiment140.zip
  inflating: /content/sentiment_data/training.1600000.processed.noemoticon.csv  


In [9]:
# checking thw size ofthe data
!ls -lh /content/sentiment_data

total 228M
-rw-r--r-- 1 root root 228M Sep 21  2019 training.1600000.processed.noemoticon.csv


In [10]:
# Checking the number of rows
!wc -l /content/sentiment_data/training.1600000.processed.noemoticon.csv

1600000 /content/sentiment_data/training.1600000.processed.noemoticon.csv


Importing the Dependencies

In [11]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [12]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
# Printing the stop words
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data Processing

In [14]:
# Loading the Dataset to the Pandas Dataframe

import pandas as pd

twitter_data = pd.read_csv('/content/sentiment140.zip'
,encoding = 'ISO-8859-1'
)

In [16]:
# Checking the number of Rows and Columns

twitter_data.shape

(1599999, 6)

In [17]:
# Now i wanna print first five rows
twitter_data.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [19]:
# Naming the columns and reading againg
column_names =['target', 'ids', 'date', 'flag', 'user', 'text']
twitter_data = pd.read_csv('/content/sentiment140.zip', names = column_names, encoding = 'ISO-8859-1')

twitter_data.shape
twitter_data.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [20]:
# Now Counting the rows which have missing values or text in the dataset

twitter_data.isnull().sum()

,0
target,0
ids,0
date,0
flag,0
user,0
text,0


In [21]:
# An other metthod to check the missing values

twitter_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 6 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   target  1600000 non-null  int64 
 1   ids     1600000 non-null  int64 
 2   date    1600000 non-null  object
 3   flag    1600000 non-null  object
 4   user    1600000 non-null  object
 5   text    1600000 non-null  object
dtypes: int64(2), object(4)
memory usage: 73.2+ MB


In [ ]:
#Now i wanna delete the missing values rows
twitter_data.dropna()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
...,...,...,...,...,...,...
1599995,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...


In [22]:
# Now verify the dtaset that rows have missing values or not

twitter_data.isnull().sum()

,0
target,0
ids,0
date,0
flag,0
user,0
text,0


In [23]:
# cheching the distribution of  target column
twitter_data['target'].value_counts()


,count
target,
0,800000
4,800000


Covert the target "4" to "1"

In [24]:
twitter_data['target'].replace({4: 1}, inplace=True)

/tmp/ipykernel_3226/985652483.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  twitter_data['target'].replace({4: 1}, inplace=True)


In [25]:
#verifying the converting of target column
twitter_data['target'].value_counts()


,count
target,
0,800000
1,800000


0 ---> Negative Tweet
    1 ---> Positive Tweet


Stemming

In [26]:
port_stem = PorterStemmer()

def stemming(content):


  stemmed_content = re.sub('[^a-zA-z]',' ',content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(words)for words in stemmed_content if not words in stopwords.words("english")]
  stemmed_content = ' '.join(stemmed_content)


  return stemmed_content



In [27]:
twitter_data["stemmed_content"] = twitter_data['text'].apply(stemming)

In [28]:
twitter_data.head()

,target,ids,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


In [29]:
print(twitter_data["stemmed_content"])

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: stemmed_content, Length: 1600000, dtype: object


In [30]:
print(twitter_data["target"])

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


In [31]:
# seprating the data and lable

x = twitter_data["stemmed_content"].values

y = twitter_data["target"].values


In [32]:
print(x)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


In [33]:
print(y)

[0 0 0 ... 1 1 1]


Spliting the Data into training data and test data

In [34]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=2)

In [36]:
print(x.shape, x_train.shape, x_test.shape)

(1600000,) (1280000,) (320000,)


In [37]:
print(x_train)

['watch saw iv drink lil wine' 'hatermagazin'
 'even though favourit drink think vodka coke wipe mind time think im gonna find new drink'
 ... 'eager monday afternoon'
 'hope everyon mother great day wait hear guy store tomorrow'
 'love wake folger bad voic deeper']


In [38]:
print(x_test)

['mmangen fine much time chat twitter hubbi back summer amp tend domin free time'
 'ah may show w ruth kim amp geoffrey sanhueza'
 'ishatara mayb bay area thang dammit' ...
 'destini nevertheless hooray member wonder safe trip' 'feel well'
 'supersandro thank']


In [39]:
# Now converting the textual data into numerical data

vectorizer = TfidfVectorizer()

x_train = vectorizer.fit_transform(x_train)
x_test =  vectorizer.transform(x_test)



In [40]:
print(x_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9416150 stored elements and shape (1280000, 483851)>
  Coords	Values
  (0, 458421)	0.27228112326228093
  (0, 372989)	0.3583819096048567
  (0, 194823)	0.5274064910069903
  (0, 116318)	0.3749338694119814
  (0, 247667)	0.4215060595161854
  (0, 464940)	0.4483581441053542
  (1, 169453)	1.0
  (2, 116318)	0.45905705320412793
  (2, 132152)	0.1891875792321668
  (2, 427967)	0.18706371586911208
  (2, 137276)	0.29069581876534506
  (2, 427034)	0.32103058232997417
  (2, 455221)	0.3296073093697297
  (2, 83506)	0.31303964332256906
  (2, 465311)	0.33480686186877023
  (2, 281186)	0.24137202118772233
  (2, 429906)	0.15168139293444055
  (2, 187474)	0.16194060015844067
  (2, 159240)	0.18800868496584808
  (2, 140239)	0.20289994039686138
  (2, 304223)	0.1678648733603879
  (3, 427034)	0.29025988953598614
  (3, 167435)	0.44570129071167713
  (3, 160331)	0.2785087322306358
  (3, 61032)	0.52009780709937
  :	:
  (1279996, 335154)	0.21190629789318696
  (

Now Training the  Machine Learning model (Logistic regression) model

In [41]:
model = LogisticRegression(max_iter = 1000)

In [42]:
model.fit(x_train, y_train)


LogisticRegression(max_iter=1000)

Model Evaluation

In [43]:
# Accuracy score on the Training data
x_training_prediction = model.predict(x_train)
training_data_accuracy = accuracy_score(y_train, x_training_prediction)


In [44]:
print("Accuracy on the Training data", training_data_accuracy)


Accuracy on the Training data 0.79812734375


In [45]:
# Accuracy score on the Test data
x_test_prediction = model.predict(x_test)
test_data_accuracy = accuracy_score(y_test, x_test_prediction)

In [46]:
print("Accuracy on the Test data", test_data_accuracy)

Accuracy on the Test data 0.777265625


Saving the trained model

In [48]:
import pickle as pkl

In [49]:
filename = "trained_model.save"

In [50]:
import pickle

pickle.dump(model, open('filename', 'wb'))